# Unsway — Phase 6C replacement-holdout baseline

This notebook rebuilds the frozen, disjoint 2,400-example holdout and evaluates **initial prompts only**. Pressure and control prompts remain unopened until the 300-example eligibility guardrail passes.

## Update the repository and install dependencies

In [ ]:
from pathlib import Path

repo = Path("/content/Unsway")
if (repo / ".git").is_dir():
    !git -C /content/Unsway pull --ff-only
else:
    !git clone https://github.com/idris404/Unsway.git /content/Unsway
%cd /content/Unsway
!pip install -q -e '.[dev]'

## Verify the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print(torch.cuda.get_device_name(0))

## Rebuild and verify both frozen datasets

Phase 6 v2 is rebuilt only so all of its questions can be excluded from the Phase 6C replacement holdout.

In [ ]:
!python -m unsway.cli.phase1 --config configs/phase1.yaml
!python -m unsway.cli.phase6 --config configs/phase6.yaml --stage data
!python -m unsway.cli.phase6 --config configs/phase6c.yaml --stage data

In [ ]:
import json

manifest = json.loads(Path("reports/phase6c_dataset.json").read_text())
assert manifest["report"]["examples"] == 2400
assert manifest["report"]["split_counts"] == {"test": 2400}
assert manifest["report"]["source_counts"] == {
    "ai2_arc": 800,
    "commonsense_qa": 800,
    "openbookqa": 800,
}
print("Protocol:", manifest["protocol_sha256"])
print("Dataset:", manifest["dataset_sha256"])
print("Replacement holdout verified.")

## Run the one-shot initial-only baseline

The runner refuses to overwrite an existing Phase 6C test result. Do not rerun this cell after it completes.

In [ ]:
!python -m unsway.cli.phase6 --config configs/phase6c.yaml --stage baseline

## Inspect the eligibility result

In [ ]:
baseline = json.loads(Path("reports/phase6c_baseline.json").read_text())
test_initial = baseline["test_initial_only"]
print(json.dumps(test_initial["metrics"], indent=2))
print("Status:", baseline["status"])
assert test_initial["pressure_scored"] is False
assert test_initial["control_scored"] is False
assert baseline["status"] == "ready_for_frozen_test", baseline["status"]
print("Eligibility passed; pressure/control test prompts remain unopened.")

## Back up and download the report

In [ ]:
import shutil

from google.colab import drive, files

drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/Unsway/phase6c")
backup.mkdir(parents=True, exist_ok=True)
for source in [
    Path("data/processed/phase6c_test_initial_predictions.jsonl"),
    Path("reports/phase6c_baseline.json"),
]:
    shutil.copy2(source, backup / source.name)
print("Backed up to", backup)
files.download("reports/phase6c_baseline.json")